In [0]:
DECLARE OR REPLACE VARIABLE catalog_use STRING DEFAULT :catalog_use;
DECLARE OR REPLACE VARIABLE schema_use STRING DEFAULT :schema_use;

USE IDENTIFIER(catalog_use || '.' || schema_use);
SELECT current_catalog(), current_schema();

In [0]:
%skip
DROP TABLE IF EXISTS fhir_bundle_zerobus;

In [0]:
%skip
CREATE TABLE himss.redox.fhir_bundle_zerobus (
  bundle_uuid STRING NOT NULL,
  fhir VARIANT,
  source_system STRING,
  event_timestamp TIMESTAMP,
  user_email STRING
)
USING DELTA
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.columnMapping.mode' = 'name',
  'description' = 'FHIR bundles ingested via Zerobus protobuf streaming'
)

In [0]:
CREATE TABLE IF NOT EXISTS fhir_bundle_zerobus
(
  bundle_uuid STRING NOT NULL COMMENT 'Unique identifier for the FHIR bundle (primary key)',
  fhir VARIANT COMMENT 'FHIR bundle payload stored as VARIANT for flexible JSON querying',
  source_system STRING COMMENT 'Source system identifier for the FHIR message',
  event_timestamp TIMESTAMP COMMENT 'Original event timestamp from the source system',
  -- user_email STRING COMMENT 'User email address associated with the FHIR message'--,
  CONSTRAINT fhir_zerobus_pk PRIMARY KEY (bundle_uuid)
)
USING DELTA
COMMENT 'Target table for zerobus FHIR bundle streaming data with VARIANT JSON storage'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true',
  'delta.feature.variantType-preview' = 'supported',
  'delta.minReaderVersion' = '3',
  'delta.minWriterVersion' = '7',
  'delta.columnMapping.mode' = 'name',
  'quality' = 'bronze',
  'pipeline' = 'fhir_zerobus',
  'description' = 'Zerobus streaming target table for FHIR bundles'
);

In [0]:
-- OPTIMIZATION STRATEGY FOR fhir_bundle_zerobus
-- Since zerobus doesn't support liquid clustering (CLUSTER BY AUTO), use traditional Delta optimization:

-- 1. Z-ORDER by query filter columns (run periodically during low-activity periods)
--    Recommended Z-ORDER columns based on likely query patterns:
--    - event_timestamp: for time-range queries
--    - source_system: for filtering by data source
--    - user_email: if filtering by user (optional)

-- Run Z-ORDER optimization
OPTIMIZE fhir_bundle_zerobus 
ZORDER BY (event_timestamp, source_system);

-- 2. Enable Predictive Optimization (requires Unity Catalog)
--    First, enable at schema or catalog level via UI or SQL:
--    ALTER SCHEMA redox ENABLE PREDICTIVE OPTIMIZATION;
--    
--    Then configure table for rewrites:
--    ALTER TABLE fhir_bundle_zerobus 
--    SET TBLPROPERTIES ('delta.tuneFileSizesForRewrites' = 'true');

-- 3. Auto-optimize settings (already configured in table properties):
--    - delta.autoOptimize.optimizeWrite = 'true' (write optimization)
--    - delta.autoOptimize.autoCompact = 'true' (small file handling)

-- 4. Data skipping automatically enabled via PRIMARY KEY (bundle_uuid)

-- Note: The VARIANT column (fhir) cannot be Z-ORDERed, but metadata columns provide efficient filtering

In [0]:
SHOW CREATE TABLE fhir_bundle_zerobus;

In [0]:
DECLARE OR REPLACE VARIABLE spn_application_id STRING;

SET VARIABLE spn_application_id = :spn_application_id;

SELECT spn_application_id;

In [0]:
DECLARE OR REPLACE grnt_stmnt STRING DEFAULT "GRANT MODIFY, SELECT ON TABLE fhir_bundle_zerobus TO `" ||spn_application_id || "`;";

SELECT grnt_stmnt;

In [0]:
EXECUTE IMMEDIATE grnt_stmnt;

In [0]:
SHOW GRANTS ON TABLE fhir_bundle_zerobus;